# Stage 1, HW 1 — the log parser

Goal: turn the raw OSF logs (Tabor, Smith & Dankowicz, Slider Game) into clean, sign-corrected, topology-labeled runs.

The parsing code is in `src/slider_logs.py`. This notebook walks through steps 1.1 – 1.5 with it.

In [1]:
import numpy as np
import pandas as pd
import networkx as nx

import config
import slider_logs as sl
from slider_logs import find, show

sl.extract_data()   # unzips the two archives in data/raw/osf/Data (only the first time)

lab_files, cloud_files = sl.log_files()
print(len(lab_files), "lab logs (2016)")
print(len(cloud_files), "cloud logs (2018)")

59 lab logs (2016)
25 cloud logs (2018)


/home/aalok/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1.1 Open one lab log and one cloud log

Two small helpers to look at pieces of a log by line number.

In [2]:
lab_file = config.LAB_DIR / "log_v09_Fri Nov 04 155722 EDT 2016.txt"
cloud_file = config.CLOUD_DIR / "Log_2018-05-31_134403.txt"

lab_lines = lab_file.read_text().splitlines()
cloud_lines = cloud_file.read_text().splitlines()

### Lab log (2016)

The file starts with the settings (XML). Each wave has its adjacency matrix and its `Weights` row there.

In [3]:
show(lab_lines, find(lab_lines, "<WaveDescription"), 14)

    67 | 	<WaveDescription Number="1">
    68 | 		<WaveSetting trialLimit="3" timeLimitSecs="300" winningValuePercentage="99"/>
    69 | 		<Adjacency> <!-- CTHierarchy7 -->
    70 | 			<row>1 1 0 0 0 1 0</row>
    71 | 			<row>1 1 0 0 0 1 0</row>
    72 | 			<row>0 0 1 1 1 0 0</row>
    73 | 			<row>0 0 1 1 1 0 0</row>
    74 | 			<row>0 0 1 1 1 0 1</row>
    75 | 			<row>1 1 0 0 0 1 1</row>
    76 | 			<row>0 0 0 0 1 1 1</row>
    77 | 		</Adjacency>
    78 | 		<Weights>-1 -1 1 1 -1 -1 1</Weights>
    79 | 		<InitialValues>0 0 0 0 0 0 0</InitialValues>
    80 | 	</WaveDescription>


The data start after `Setting File End`. A run starts at `Wave Starting: k`.
Then the rows come in pairs: **time + 7 raw positions**, and below it **7 Agreements + General-Success**.

In [4]:
show(lab_lines, find(lab_lines, "Wave Starting: 1"), 11)

   230 | Wave Starting: 1
   231 | Players:   		1   	2   	3   	4   	5   	6   	7   	General-Success
   232 | 0.16 seconds	0.00	0.00	0.00	0.00	0.00	0.00	0.00
   233 |             	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00
   234 | 
   235 | 3.04 seconds	0.00	0.00	-0.10	0.00	0.00	0.00	0.00
   236 |             	0.00	0.00	3.33	3.33	2.50	0.00	0.00	1.43
   237 | 
   238 | 3.34 seconds	0.00	0.10	-0.10	0.00	0.00	0.00	0.00
   239 |             	3.33	3.33	3.33	3.33	2.50	2.50	0.00	2.86
   240 | 


The run ends at `Round Success` (or `Round Failed`), and the next run starts on the next line.

In [5]:
show(lab_lines, find(lab_lines, "Round Success") - 3, 7)

  2146 | 111.04 seconds	1.00	1.00	-1.00	-1.00	1.00	1.00	-1.00
  2147 |               	100.00	100.00	100.00	100.00	100.00	100.00	100.00	100.00
  2148 | 
  2149 | Round Success
  2150 | Wave Starting: 2
  2151 | Players:   		1   	2   	3   	4   	5   	6   	7   	General-Success
  2152 | 0.06 seconds	0.00	0.00	0.00	0.00	0.00	0.00	0.00


### Cloud log (2018)

Every line has a `HH:MM:SS.mmm:` clock in front. A run starts at `Starting Game k`, followed by the adjacency matrix and the `Weights` row.

In [6]:
show(cloud_lines, find(cloud_lines, "Starting Game 1"), 12)

   334 | 00:02:55.319: Starting Game 1
   335 | 00:02:55.319: Adjacency Matrix: wagonwheel-6
   336 | 00:02:55.319: 1 0 0 0 0 1 0
   337 | 00:02:55.319: 0 1 0 0 0 1 0
   338 | 00:02:55.319: 0 0 1 0 0 1 0
   339 | 00:02:55.319: 0 0 0 1 0 1 0
   340 | 00:02:55.319: 0 0 0 0 1 1 0
   341 | 00:02:55.319: 1 1 1 1 1 1 1
   342 | 00:02:55.319: 0 0 0 0 0 1 1
   343 | 00:02:55.319: Weights: -1.0 -1.0 1.0 1.0 -1.0 -1.0 1.0
   344 | 00:02:55.319: InitialValues: 0.0 0.0 0.0 0.0 0.0 0.0 0.0
   345 | 00:02:55.319: Resetting timer for wave timeout of 180 seconds


Each play gives an `Adjusted Plays` row (**7 raw positions**) and an `Extra:` row (**7 Agreements + overall**).

In [7]:
show(cloud_lines, find(cloud_lines, "received play of '0.1'"), 3)

   485 | 00:03:02.605: Extra: received play of '0.1' from player 1
   486 | 00:03:02.605: Adjusted Plays: 0.1  0.0  0.0  0.0  0.0  0.0  0.0
   487 | 00:03:02.605: Extra: 	5.00	0.00	0.00	0.00	0.00	1.43	0.00	1.43


When a player drops out, a robot takes over. Robot positions are written in parentheses.

In [8]:
show(cloud_lines, find(cloud_lines, "converted player"), 8)

   789 | 00:03:13.180: converted player 5 to robot because PingbackTimeout
   790 | 00:03:13.206: Extra: received play of '0.0' from player 3
   791 | 00:03:13.212: Extra: received play of '1.0' from player 1
   792 | 00:03:13.221: Extra: received play of '1.0' from player 7
   793 | 00:03:13.222: Extra: received play of '0.0' from player 6
   794 | 00:03:13.236: A(nother) robot appeared: robot<--source map is now: 5<--6
   795 | 00:03:13.236: Adjusted Plays: 1.0  0.5  0.0  0.0  (0.0)  0.0  1.0
   796 | 00:03:13.236: Extra: 	50.00	25.00	0.00	0.00	0.00	7.14	50.00	7.14


The run ends at `Wave timed out` (failure) or `Round Success`.

In [9]:
show(cloud_lines, find(cloud_lines, "Wave timed out") - 3, 5)
print()
show(cloud_lines, find(cloud_lines, "Round Success") - 2, 4)

  6839 | 00:05:55.055: Extra: received play of '0.5' from player 2
  6840 | 00:05:55.055: Adjusted Plays: 1.0  0.5  -1.0  -1.0  (1.0)  1.0  -1.0
  6841 | 00:05:55.055: Extra: 	100.00	75.00	100.00	100.00	100.00	92.86	100.00	92.86
  6842 | 00:05:55.975: Wave timed out
  6843 | 00:05:55.975: there's another game to be played

 10767 | 00:08:24.242: Adjusted Plays: (-1.0)  1.0  -1.0  1.0  (-1.0)  1.0  -1.0
 10768 | 00:08:24.242: Extra: 	100.00	100.00	100.00	100.00	100.00	100.00	100.00	100.00
 10769 | 00:08:24.242: Round Success
 10770 | 00:08:24.242: there's another game to be played


### Summary of 1.1

| Item | Lab log (2016) | Cloud log (2018) |
|---|---|---|
| Timestamp | `3.04 seconds` at the start of a positions row. Seconds since the wave started. | `HH:MM:SS.mmm:` at the start of every line. Session clock, so subtract the time of `Starting Game k`. |
| 7 positions | Same row as the timestamp. Raw (not sign-corrected), in [−1, 1]. | `Adjusted Plays:` row. Raw. `(x)` = robot. |
| Logged Agreements | Next row (starts with a tab): 7 percentages + General-Success. | `Extra: <tab>…` row right after `Adjusted Plays`: 7 percentages + overall. |
| `Weights` row | `<Weights>` inside the wave's `<WaveDescription>` in the settings. | `Weights:` line after `Starting Game k`. |
| Adjacency matrix | `<row>` lines inside `<Adjacency>` of the same block. | The 7 lines after `Adjacency Matrix: name`. |
| Run starts | `Wave Starting: k` | `Starting Game k` |
| Run ends | `Round Success` / `Round Failed` | `Round Success` / `Wave timed out` |

In both formats the logged adjacency matrix has **1s on the diagonal** (each player counts itself), and rows are written when something happens, so they are **not evenly spaced in time**.

## 1.2 Parse one run

`parse_lab` / `parse_cloud` return a list of runs (one dict per run). Take the first run of the lab log.

In [10]:
lab_runs = sl.parse_lab(lab_file)
run = lab_runs[0]

A = run["A"]              # 7x7, diagonal removed
weights = run["weights"]  # 7
t = run["t"]              # T
X = run["X"]              # T x 7, raw positions
agree = run["agree"]      # T x 7, logged Agreement

print("topology name in the log:", run["name"], "| end:", run["end"])
print("A", A.shape, " weights", weights.shape, " t", t.shape, " X", X.shape, " agree", agree.shape)
print()
print("A =")
print(A)
print("weights =", weights)
print()
print("first 5 rows")
print("t     =", t[:5])
print("X     =")
print(X[:5])
print("agree =")
print(agree[:5])

topology name in the log: CTHierarchy7 | end: success
A (7, 7)  weights (7,)  t (639,)  X (639, 7)  agree (639, 7)

A =
[[0 1 0 0 0 1 0]
 [1 0 0 0 0 1 0]
 [0 0 0 1 1 0 0]
 [0 0 1 0 1 0 0]
 [0 0 1 1 0 0 1]
 [1 1 0 0 0 0 1]
 [0 0 0 0 1 1 0]]
weights = [-1. -1.  1.  1. -1. -1.  1.]

first 5 rows
t     = [0.16 3.04 3.34 3.84 4.54]
X     =
[[ 0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.  -0.1  0.   0.   0.   0. ]
 [ 0.   0.1 -0.1  0.   0.   0.   0. ]
 [ 0.1  0.1 -0.2  0.   0.   0.   0. ]
 [ 0.1  0.1 -0.3  0.   0.   0.   0. ]]
agree =
[[ 0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    3.33  3.33  2.5   0.    0.  ]
 [ 3.33  3.33  3.33  3.33  2.5   2.5   0.  ]
 [ 6.67  6.67  6.67  6.67  5.    5.    0.  ]
 [ 6.67  6.67 10.   10.    7.5   5.    0.  ]]


The same for one cloud run (game 2 of the cloud log).

In [11]:
cloud_runs = sl.parse_cloud(cloud_file)
crun = cloud_runs[1]

print("topology name in the log:", crun["name"], "| end:", crun["end"], "| bot:", crun["bot"])
print("A", crun["A"].shape, " weights", crun["weights"].shape, " t", crun["t"].shape,
      " X", crun["X"].shape, " agree", crun["agree"].shape)
print("weights =", crun["weights"])
print("last row:  t =", crun["t"][-1], " X =", crun["X"][-1])

topology name in the log: hierarchy-7 | end: success | bot: True
A (7, 7)  weights (7,)  t (1268,)  X (1268, 7)  agree (1268, 7)
weights = [-1.  1. -1.  1. -1.  1. -1.]
last row:  t = 146.48900000000003  X = [-1.  1. -1.  1. -1.  1. -1.]


## 1.3 Sign-correct and recompute Agreement

Sign-corrected positions: `s = weights ⊙ X`.

Agreement of player `i` (percent), with `N(i)` = player `i` and its neighbours:

$$\text{Agreement}_i = 100 \cdot \frac{\left|\sum_{j \in N(i)} s_j\right|}{|N(i)|}$$

In matrix form this is `100 * |A_log s| / rowsum(A_log)`, where `A_log` is the logged matrix **with** the 1s on the diagonal.

The log is rounded to 2 decimals, so "max error ≈ 0" means at most 0.005.

In [12]:
s = weights * X                     # sign-corrected positions, T x 7

A_log = run["A_log"]                # adjacency as logged (diagonal = 1)
recomputed = 100 * np.abs(s @ A_log.T) / A_log.sum(axis=1)

print("lab run:   max |recomputed - logged| =", np.abs(recomputed - agree).max())

# same check for the cloud run, using the functions in slider_logs.py
cs = sl.sign_correct(crun)
print("cloud run: max |recomputed - logged| =", np.abs(sl.agreement(crun["A_log"], cs) - crun["agree"]).max())

lab run:   max |recomputed - logged| = 0.0033333333333445125
cloud run: max |recomputed - logged| = 0.0033333333333445125


The sign correction is what makes the final state readable: in a won run the raw positions `X` have mixed signs, but `s` is all +1 or all −1.

In [13]:
print("end:", run["end"])
print("last X =", X[-1])
print("last s =", s[-1])

end: success
last X = [ 1.  1. -1. -1.  1.  1. -1.]
last s = [-1. -1. -1. -1. -1. -1. -1.]


## 1.4 Label the topology by degree sequence

The logs relabel the players from wave to wave (for example `Hierarchy1`, `Hierarchy3`, `Hierarchy7` put the root at player 1, 3 or 7), so the matrix itself changes. The **sorted degree sequence** does not.

By hand (7 players, no self-loops):

| Topology | Structure | Edges | Sorted degree sequence |
|---|---|---|---|
| Full | everyone linked to the other 6 | 21 | 6 6 6 6 6 6 6 |
| Star | 1 hub linked to 6 leaves | 6 | 1 1 1 1 1 1 6 |
| Hierarchy | root with 2 children, each with 2 leaves: root 2, middle 3, leaf 1 | 6 | 1 1 1 1 2 3 3 |
| CT-Hierarchy | Hierarchy + a link between the two leaves of each middle node: leaves go from 1 to 2 | 8 | 2 2 2 2 2 3 3 |
| Ring | everyone has 2 neighbours | 7 | 2 2 2 2 2 2 2 |

All five are different, so the degree sequence is enough to tell them apart. Check with NetworkX:

In [14]:
by_hand = {
    "Full":         (6, 6, 6, 6, 6, 6, 6),
    "Star":         (1, 1, 1, 1, 1, 1, 6),
    "Hierarchy":    (1, 1, 1, 1, 2, 3, 3),
    "CT-Hierarchy": (2, 2, 2, 2, 2, 3, 3),
    "Ring":         (2, 2, 2, 2, 2, 2, 2),
}

hierarchy = [(1, 2), (1, 3), (2, 4), (2, 5), (3, 6), (3, 7)]
edges = {
    "Full":         [(i, j) for i in range(1, 8) for j in range(i + 1, 8)],
    "Star":         [(1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7)],
    "Hierarchy":    hierarchy,
    "CT-Hierarchy": hierarchy + [(4, 5), (6, 7)],
    "Ring":         [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 1)],
}

for name, e in edges.items():
    G = nx.Graph(e)
    degrees = tuple(sorted(d for node, d in G.degree()))
    print(f"{name:13s} {degrees}  matches by hand: {degrees == by_hand[name]}")

print("all distinct:", len(set(by_hand.values())) == 5)
print("same table as slider_logs.DEGREE_SEQUENCES:",
      sl.DEGREE_SEQUENCES == {degrees: name for name, degrees in by_hand.items()})

Full          (6, 6, 6, 6, 6, 6, 6)  matches by hand: True
Star          (1, 1, 1, 1, 1, 1, 6)  matches by hand: True
Hierarchy     (1, 1, 1, 1, 2, 3, 3)  matches by hand: True
CT-Hierarchy  (2, 2, 2, 2, 2, 3, 3)  matches by hand: True
Ring          (2, 2, 2, 2, 2, 2, 2)  matches by hand: True
all distinct: True
same table as slider_logs.DEGREE_SEQUENCES: True


Label the two runs from 1.2 and compare with the name written in the log.

In [15]:
for r in [run, crun]:
    degrees = sorted(r["A"].sum(axis=1).tolist())
    print("log name:", r["name"], "| degrees:", degrees, "| label:", sl.label_topology(r["A"]))

log name: CTHierarchy7 | degrees: [2, 2, 2, 2, 2, 3, 3] | label: CT-Hierarchy
log name: hierarchy-7 | degrees: [1, 1, 1, 1, 2, 3, 3] | label: Hierarchy


## 1.5 One table of runs from all files

Columns:

- `file` — the log file (one file = one group of players)
- `session` — `lab` (2016) or `cloud` (2018)
- `dataset` — `EL`, `CL` or `CC`, following the OSF scripts `selectdatasetEL.m`, `selectdatasetCL.m`, `selectdatasetCC.m`; empty if the file is in none of them
- `wave` — position of the run in the session (1, 2, …)
- `players` — number of players
- `topology` — label from the degree sequence (`other` if it is not one of the five)
- `success` — True if the run ended with `Round Success`
- `end` — `success`, `failed` or `incomplete` (no end marker in the log)
- `duration` — time of the last logged row, in seconds from the start of the run
- `bot` — True if a robot played in the run
- `T` — number of logged rows
- `agree_error` — max |recomputed − logged Agreement| over the run (step 1.3)

In [16]:
runs = sl.parse_all()
table = sl.runs_table(runs)

print(len(table), "runs from", table["file"].nunique(), "files")
table

718 runs from 83 files


,file,session,dataset,wave,players,topology,success,end,duration,bot,T,agree_error
0,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,1,7,Ring,False,failed,300.000,False,2056,0.003333
1,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,2,7,Hierarchy,True,success,59.000,False,470,0.003333
2,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,3,7,Full,True,success,90.000,False,734,0.004286
3,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,4,7,Hierarchy,True,success,80.000,False,730,0.003333
4,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,5,7,Full,True,success,21.000,False,156,0.004286
...,...,...,...,...,...,...,...,...,...,...,...,...
713,Log_2018-05-31_183920.txt,cloud,CC,5,7,Star,True,success,11.343,False,222,0.004286
714,Log_2018-05-31_183920.txt,cloud,CC,6,7,Full,True,success,105.953,False,1370,0.004286
715,Log_2018-05-31_183920.txt,cloud,CC,7,7,Hierarchy,True,success,18.812,False,268,0.003333
716,Log_2018-05-31_183920.txt,cloud,CC,8,7,Full,True,success,17.970,False,221,0.004286


### Checks

Step 1.3 for every run: the recomputed Agreement matches the logged one everywhere.

In [17]:
print("max agreement error over all runs:", table["agree_error"].max())
print("runs with no logged rows:", (table["T"] == 0).sum())

max agreement error over all runs: 0.00428571428574287
runs with no logged rows: 8


Step 1.4 for every run: degree-sequence label against the topology name written in the log (where the log has one). The trailing number in the log name is only the player relabeling, so it is removed. `Circle` = Ring, `wagonwheel` = Star.

In [18]:
log_name = pd.Series([r["name"] for r in runs]).str.replace(r"-?\d+$", "", regex=True)
log_name = log_name.replace("", "(no name)")
pd.crosstab(log_name, table["topology"], rownames=["name in log"])

topology,CT-Hierarchy,Full,Hierarchy,Ring,Star,other
name in log,,,,,,
(no name),9,93,99,78,21,26
CTHierarchy,51,0,0,0,0,0
Circle,0,0,0,51,0,6
Full,0,0,0,0,0,6
Hierarchy,0,0,51,0,0,6
full,0,74,0,0,0,0
hierarchy,0,0,74,0,0,0
wagonwheel,0,0,0,0,73,0


The `other` rows are the 4-player and 15-player sessions, plus three 7-player sessions (Oct 26) whose settings contain two malformed matrices each (one non-symmetric, one hierarchy with a missing link). None of them belongs to EL, CL or CC:

In [19]:
pd.crosstab([table["session"], table["dataset"], table["players"]], table["topology"])

topology                 CT-Hierarchy  Full  Hierarchy  Ring  Star  other
session dataset players                                                  
cloud   CC      7                   0    74         74     0    73      0
lab             4                   0     0          0     0     0     18
                7                   9    30         36    17    21      6
                15                  0     0          0     0     0     20
        CL      7                   0    63         63    61     0      0
        EL      7                  51     0         51    51     0      0

How the runs ended:

In [20]:
pd.crosstab([table["session"], table["dataset"]], table["end"])

end              failed  incomplete  success
session dataset                             
cloud   CC            7          17      197
lab                  36           7      114
        CL           22           2      163
        EL           18           0      135

### Runs per dataset and topology

Only the three datasets of the paper, and only runs that have an end marker.

In [21]:
good = table[(table["dataset"] != "") & (table["end"] != "incomplete")]

summary = good.groupby(["dataset", "topology"]).agg(
    runs=("success", "size"),
    success=("success", "sum"),
    with_bot=("bot", "sum"),
    median_duration=("duration", "median"),
)
summary

runs  success  with_bot  median_duration
dataset topology                                              
CC      Full            66       64        30          39.0015
        Hierarchy       70       66        30          31.4795
        Star            68       67        29          29.2715
CL      Full            63       52         0          89.1900
        Hierarchy       63       58         0          45.1000
        Ring            59       53         0          60.6200
EL      CT-Hierarchy    51       42         0          51.5700
        Hierarchy       51       48         0          36.2800
        Ring            51       45         0          55.9500

### Save

In [22]:
table.to_csv(config.RUNS_TABLE, index=False)

## Things found in the logs

- **Wave numbering (lab).** Many lab settings files number the 9th wave `8` again, and the log then prints `Wave Starting: 8` twice. The parser therefore matches the k-th `Wave Starting` to the k-th wave description, and `wave` in the table is that position (1 – 9).
- **Duplicate file (cloud).** `tmp.txt` is a byte-for-byte copy of `Log_2018-05-31_132916.txt`. It is skipped.
- **Incomplete runs.** `end = incomplete` means the log has no end marker. In the cloud logs these are mostly games that the server skipped right after starting them (0 – 12 rows), or a session that hit its time limit. In the lab logs the session was closed mid-wave.
- **Three 1-row "failed" runs (lab).** The 9th wave of three CL sessions is marked `Round Failed` after a single row (`T = 1`). They look like the session being closed, not a real failure.
- **`duration`** is the time of the last logged row. Rows are only written when something happens, so a lab run that timed out at 300 s can show a little less.
- **Robots** only exist in the cloud logs. `bot` is True if any position in the run was written in parentheses.